# Git bisect & history rewrite: interactive walkthrough> last_verified: 2026-08-01 · gitThis notebook automates two git skills that come up when you need to track down aregression after the fact and then clean up the history around it:- **git bisect** — binary-searches your history to find the first commit that broke a  test, driven by a test script via `git bisect run`.- **history rewrite** — drops the bad commit and amends messages with `git rebase -i`  (driven non-interactively through `GIT_SEQUENCE_EDITOR`), plus the `git reflog` safety  net to roll back when a rewrite goes wrong.I build a throwaway repo in `/tmp` with a deliberately broken `add(a, b)` (returns `a * b`instead of `a + b`) and exercise every command against it. Nothing here touches your realrepositories — the demo lives only in `/tmp/git-bisect-demo`.</markdown_cell>

## What I'll buildA five-commit history ending on a buggy tip, plus a small `run_tests.sh` that exits 0 when`add(2, 3) == 5` and non-zero otherwise — the exact signal `git bisect run` needs. Thegood commit is `HEAD~3` ("add correct add() implementation"); the tip (`HEAD`) is buggy.The workflow mirrors what the docs keep recommending: mark the buggy tip and a known-goodancestor, let bisect walk the commits, then lean on the reflog when a rebase rewrite needsto be undone .</markdown_cell>

## Setup — create a reproducible demo repo with a known bug</markdown_cell>

In [ ]:
%%bashPROJECT_DIR=/tmp/git-bisect-demorm -rf "$PROJECT_DIR"mkdir -p "$PROJECT_DIR"cd "$PROJECT_DIR"git init -qgit config user.email "agent06@example.com"git config user.name "Agent 06"git config commit.gpgsign false# 1) root commitprintf '# demo\nA throwaway repo for git bisect + history-rewrite practice.\n' > README.mdgit add README.mdgit commit -q -m "init demo repo"# 2) correct implementationcat > calc.py <<'PY'def add(a, b):    return a + bPYgit add calc.pygit commit -q -m "add correct add() implementation"# 3) test runner: exit 0 when add(2,3)==5, non-zero otherwise.#    -B and clearing __pycache__ keep the result deterministic across the#    rapid checkouts that git bisect performs (see "What I ran into").cat > run_tests.sh <<'SH'#!/usr/bin/env bash# Exit 0 = "good" (add(2,3) == 5), non-zero = "bad" (bug present).rm -rf __pycache__python3 -B - <<'PY'import importlib.utilspec = importlib.util.spec_from_file_location("calc", "calc.py")m = importlib.util.module_from_spec(spec)spec.loader.exec_module(m)assert m.add(2, 3) == 5, "add(2,3) should be 5"print("add(2,3) =", m.add(2, 3), "-> good")PYSHchmod +x run_tests.shgit add run_tests.shgit commit -q -m "add test runner for add()"# 4) BUG: add() now returns the productcat > calc.py <<'PY'def add(a, b):    return a * bPYgit commit -q -am "introduce bug in add() returns product"# 5) a harmless follow-up (docstring) on top of the bug — still brokencat > calc.py <<'PY'"""demo module"""def add(a, b):    return a * bPYgit commit -q -am "add module docstring"echo "Demo repo ready at $PROJECT_DIR"echo "=== history ==="git log --onelineecho "=== bug check (expect FAIL) ==="./run_tests.sh && echo "PASS" || echo "FAIL — bug present"</code_cell>

## Step 1 — Find the first bad commit with `git bisect run`I mark the tip as bad and the known-good ancestor (`HEAD~3`) as good, then let`run_tests.sh` drive the binary search.</markdown_cell>

In [ ]:
%%bashPROJECT_DIR=/tmp/git-bisect-democd "$PROJECT_DIR"GOOD_REF="$(git rev-parse --short HEAD~3)"echo "good (known-correct): $GOOD_REF"git bisect reset 2>/dev/nullgit bisect startgit bisect bad HEADgit bisect good "$GOOD_REF"echo "=== git bisect run ==="git bisect run ./run_tests.shecho "=== first bad commit ==="git bisect log | grep -i "first bad commit"git bisect reset 2>/dev/null</code_cell>

## Step 2 & 3 — Rewrite history, then roll back with `git reflog`The cleanest rewrite for a commit you want gone is `git rebase -i` with that line dropped.Because the rebase editor is interactive, I drive it non-interactively with`GIT_SEQUENCE_EDITOR`: a `sed` flips `pick` → `drop` on the todo line whose message contains"introduce bug". The docstring-only follow-up applies cleanly on the good base, so the bugdisappears and `run_tests.sh` passes again.Once rewritten, the original buggy tip isn't lost — it's only unreachable, and `git reflog`will find it. I capture the pre-rewrite SHA, drop the bug, verify it's gone, roll back tothe captured SHA to prove recovery, then re-apply the rewrite so the repo ends clean.</markdown_cell>

In [ ]:
%%bashPROJECT_DIR=/tmp/git-bisect-democd "$PROJECT_DIR"# Capture the buggy tip so we can roll back to it.BEFORE="$(git rev-parse HEAD)"echo "tip before rewrite: $BEFORE"# Drop the bug commit non-interactively.GIT_SEQUENCE_EDITOR="sed -i '/introduce bug/s/^pick/drop/'" git rebase -i HEAD~2 2>&1 | tail -1echo "=== history after dropping the bug commit ==="git log --onelineecho "=== verify (expect PASS — bug is gone) ==="./run_tests.sh && echo "PASS — bug removed" || echo "FAIL"echo "=== rollback: reflog (last 3) then reset --hard ==="git reflog -3 --onelinegit reset --hard "$BEFORE" >/dev/null 2>&1echo "=== verify after rollback (expect FAIL — bug restored) ==="./run_tests.sh && echo "PASS" || echo "FAIL — bug restored (original history is back)"# Re-apply the rewrite so the repo ends in a clean, fixed state.GIT_SEQUENCE_EDITOR="sed -i '/introduce bug/s/^pick/drop/'" git rebase -i HEAD~2 2>&1 | tail -1echo "=== re-applied rewrite (expect PASS) ==="./run_tests.sh && echo "PASS — clean end state" || echo "FAIL"</code_cell>

## Step 4 — Amend the last commit messageA common history-rewrite move is rewriting the most recent commit's message. `git commit--amend` swaps the top commit in place — for a commit in the middle of history, `gitrebase -i` with `reword` is the equivalent.</markdown_cell>

In [ ]:
%%bashPROJECT_DIR=/tmp/git-bisect-democd "$PROJECT_DIR"git commit --amend -m "add module docstring explaining the demo"echo "=== final history ==="git log --onelineecho "=== final verify (expect PASS) ==="./run_tests.sh && echo "PASS" || echo "FAIL"</code_cell>

## What I ran into- **Stale `.pyc` between bisect checkouts.** When `run_tests.sh` first loaded `calc.py`  with `importlib`, Python cached a `calc.cpython-*.pyc`. Because `git bisect` checks out  commits in rapid succession (often within the same second), the bytecode from a buggy  checkout shadowed the correct `calc.py` on the next checkout — bisect then blamed a  good commit. Running with `python3 -B` and `rm -rf __pycache__` before each check fixed it.- **`git bisect run` leaves HEAD on the last-tested commit, not the first-bad one.** After  `run` finishes, `git log -1` can show a good commit even though bisect *reported* the  correct first bad commit. I read the answer from `git bisect log | grep "first bad"` in  Step 1 rather than trusting the post-run `HEAD`.</markdown_cell>

## What I'd try next- Swap `run_tests.sh` for a real test command (`pytest`, `go test`, `npm test`) and run it  against a repo with 20–30 commits so the O(log n) savings of bisect are visible.- Try `git rebase -i --autosquash` with a `fixup!`/`squash!` flow to clean up a messy stack  in one pass, instead of hand-editing the todo list with `sed`.- Tear down the demo: `rm -rf /tmp/git-bisect-demo` when you're done.</markdown_cell>